# Backtest schedule analytics

Load a **`dates.csv` schedule** (from `scripts/backtest_mvp.py --dates-csv`) *or* an **`equity.csv`** from a run folder.

When using **`--dates-csv`**, each completed row includes **running** analysis columns:
`fees_day`, `cumulative_fees`, `total_return`, `annualized_return`, `sharpe_ratio`, `max_drawdown`, `total_transaction_costs`, `cost_bps`, `processed_signal` (same definitions as `summary.json`, recomputed through that date).

Set paths in the next cell, then run all.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

_cwd = Path.cwd().resolve()
if (_cwd / "tradingagents").is_dir():
    REPO_ROOT = _cwd
elif (_cwd.parent / "tradingagents").is_dir():
    REPO_ROOT = _cwd.parent
else:
    REPO_ROOT = _cwd

# Schedule from backtest_mvp --dates-csv (per-day equity in columns after each run)
CSV_PATH = REPO_ROOT / "eval_results" / "RELIANCE.NS" / "dates.csv"

# Optional: book ledger from a run (richer: fees_day, cumulative_fees)
EQUITY_CSV_PATH: Path | None = None  # e.g. REPO_ROOT / "eval_results" / "RELIANCE.NS" / "backtest_mvp_xxxx" / "equity.csv"

# Optional: runner summary (annualized_return, sharpe_ratio, total_transaction_costs, ...)
SUMMARY_JSON_PATH: Path | None = None  # e.g. same folder as equity.csv / "summary.json"

%matplotlib inline

try:
    from IPython.display import display
except ImportError:

    def display(obj):
        print(obj)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")


In [ ]:
def processed_mask(proc: pd.Series) -> pd.Series:
    """Row-wise: True where schedule row counts as processed (matches backtest CSV semantics)."""
    if pd.api.types.is_bool_dtype(proc):
        return proc.fillna(False)
    s = proc.fillna("").astype(str).str.strip().str.lower()
    return s.isin(("1", "true", "yes", "y"))


def load_schedule_csv(path: Path) -> pd.DataFrame:
    """Load dates schedule; keep rows valid for equity / ROI (processed, no error, numeric equity)."""
    df = pd.read_csv(path, encoding="utf-8-sig")
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["equity"] = pd.to_numeric(df["equity"], errors="coerce")
    df["close"] = pd.to_numeric(df["close"], errors="coerce")
    df["cash"] = pd.to_numeric(df["cash"], errors="coerce")
    df["shares"] = pd.to_numeric(df["shares"], errors="coerce")

    err = df["error"].fillna("").astype(str).str.strip()
    ok = df["equity"].notna() & processed_mask(df["processed"]) & (err == "")
    plot_df = df.loc[ok].sort_values("date").reset_index(drop=True)
    return plot_df


def load_equity_csv(path: Path) -> pd.DataFrame:
    """Per-day book from ``backtest_mvp`` ``equity.csv`` (all rows with valid equity)."""
    df = pd.read_csv(path, encoding="utf-8-sig")
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["equity"] = pd.to_numeric(df["equity"], errors="coerce")
    for col in ("close", "cash", "shares", "fees_day", "cumulative_fees"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    plot_df = df.loc[df["equity"].notna()].sort_values("date").reset_index(drop=True)
    return plot_df


def load_plot_frame() -> pd.DataFrame:
    if EQUITY_CSV_PATH is not None and Path(EQUITY_CSV_PATH).is_file():
        return load_equity_csv(Path(EQUITY_CSV_PATH))
    return load_schedule_csv(CSV_PATH)


plot_df = load_plot_frame()
if plot_df.empty:
    raise ValueError("No rows to plot — check CSV_PATH / EQUITY_CSV_PATH")

initial_equity = float(plot_df["equity"].iloc[0])
plot_df = plot_df.copy()
plot_df["roi_cum"] = plot_df["equity"] / initial_equity - 1.0
plot_df["roi_pct"] = 100.0 * plot_df["roi_cum"]
plot_df["period_return"] = plot_df["equity"].pct_change()
peak = plot_df["equity"].cummax()
plot_df["drawdown"] = (plot_df["equity"] - peak) / peak

plot_df.head(10)


In [ ]:
final_eq = float(plot_df["equity"].iloc[-1])
total_return = final_eq / initial_equity - 1.0
max_dd = float(plot_df["drawdown"].min())
n = len(plot_df)
retrs = plot_df["period_return"].dropna()
vol = float(retrs.std()) if len(retrs) > 1 else float("nan")
mean_r = float(retrs.mean()) if len(retrs) else float("nan")
sharpe_simple = mean_r / vol if vol and vol > 0 else float("nan")

summary = {
    "initial_equity": initial_equity,
    "final_equity": final_eq,
    "total_return_pct": 100.0 * total_return,
    "max_drawdown_pct": 100.0 * max_dd,
    "n_points": n,
    "period_return_std": vol,
    "period_return_mean": mean_r,
    "sharpe_like_mean_over_std": sharpe_simple,
}

runner_summary = {}
if SUMMARY_JSON_PATH is not None and Path(SUMMARY_JSON_PATH).is_file():
    runner_summary = json.loads(Path(SUMMARY_JSON_PATH).read_text(encoding="utf-8"))

display_cols = [
    "total_return",
    "annualized_return",
    "sharpe_ratio",
    "max_drawdown",
    "total_transaction_costs",
    "cost_bps",
    "final_equity",
    "initial_cash",
    "status",
]
if runner_summary:
    row = {k: runner_summary.get(k) for k in display_cols if k in runner_summary}
    runner_table = pd.Series(row, name="from summary.json").to_frame("value")
    print("Runner snapshot (whole-run aggregates)")
    display(runner_table)
else:
    print("Set SUMMARY_JSON_PATH to load annualized_return / sharpe_ratio / fees from the backtest run.")

print("Schedule- or equity-curve-derived (step stats; Sharpe not annualized)")
pd.Series(summary).to_frame("value")


In [ ]:
nrows = 4 if ("cumulative_fees" in plot_df.columns and plot_df["cumulative_fees"].notna().any()) else 3
fig_h = 12 if nrows == 4 else 9
fig, axes = plt.subplots(nrows, 1, figsize=(10, fig_h), sharex=True)
if nrows == 3:
    axes = [axes[0], axes[1], axes[2]]

dates = plot_df["date"]

axes[0].plot(dates, plot_df["equity"], color="C0", marker="o", markersize=3, label="Equity")
axes[0].axhline(initial_equity, color="gray", linestyle="--", linewidth=1, label="Initial")
axes[0].set_ylabel("Equity")
axes[0].legend(loc="upper left")
title_src = "equity.csv" if EQUITY_CSV_PATH and Path(EQUITY_CSV_PATH).is_file() else CSV_PATH.name
axes[0].set_title(f"Equity & ROI — {title_src}")

axes[1].fill_between(dates, 0, 100 * plot_df["roi_cum"], alpha=0.35, color="C2")
axes[1].plot(dates, 100 * plot_df["roi_cum"], color="C2", marker="o", markersize=3)
axes[1].axhline(0, color="gray", linewidth=0.8)
axes[1].set_ylabel("Cumulative ROI (%)")

axes[2].fill_between(dates, 100 * plot_df["drawdown"], 0, alpha=0.4, color="C3")
axes[2].plot(dates, 100 * plot_df["drawdown"], color="C3", marker="o", markersize=3)
axes[2].set_ylabel("Drawdown (%)")

if nrows == 4:
    axf = axes[3]
    axf.bar(dates, plot_df["fees_day"].fillna(0), width=0.8, color="C4", label="fees_day")
    axf.plot(dates, plot_df["cumulative_fees"], color="k", marker="s", markersize=3, label="cumulative_fees")
    axf.set_ylabel("Fees")
    axf.legend(loc="upper left")
    axf.set_xlabel("Date")
else:
    axes[2].set_xlabel("Date")

plt.tight_layout()
plt.show()

fig2, ax = plt.subplots(figsize=(10, 4))
if plot_df["close"].notna().any():
    ax.plot(dates, plot_df["close"], color="C1", label="Close")
    ax.set_ylabel("Close")
    ax.set_title("Close (when available)")
    ax.tick_params(axis="y", labelcolor="C1")
plt.tight_layout()
plt.show()
